# RoClaw-Distill: Fine-tune Qwen3-VL-2B on Cognitive ISA

Fine-tunes Qwen3-VL-2B-Instruct via LoRA (Unsloth) on text-scene navigation traces
so the model natively speaks the TOOLCALL motor command format.

**Prerequisites:**
- Training data (`train.jsonl`, `val.jsonl`) uploaded to Google Drive
- Colab with GPU runtime (T4 or better)

In [ ]:
# Cell 1: Install dependencies
!pip install -q unsloth transformers datasets peft trl accelerate bitsandbytes
!pip install -q llama-cpp-python  # for GGUF export

In [ ]:
# Cell 2: Mount Drive and load training data
from google.colab import drive
drive.mount('/content/drive')

import json
from pathlib import Path

# Update this path to match your Drive location
DATA_DIR = Path('/content/drive/MyDrive/roclaw-distill')
TRAIN_PATH = DATA_DIR / 'train.jsonl'
VAL_PATH = DATA_DIR / 'val.jsonl'

assert TRAIN_PATH.exists(), f'Training data not found at {TRAIN_PATH}'
assert VAL_PATH.exists(), f'Validation data not found at {VAL_PATH}'

# Quick stats
with open(TRAIN_PATH) as f:
    train_count = sum(1 for _ in f)
with open(VAL_PATH) as f:
    val_count = sum(1 for _ in f)

print(f'Training examples: {train_count}')
print(f'Validation examples: {val_count}')

In [ ]:
# Cell 3: Load model with Unsloth
from unsloth import FastLanguageModel

MODEL_NAME = 'Qwen/Qwen3-VL-2B-Instruct'
MAX_SEQ_LENGTH = 2048  # text scenes are ~500 tokens

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # auto-detect
    load_in_4bit=True,  # save VRAM during training
)

print(f'Model loaded: {MODEL_NAME}')
print(f'Parameters: {model.num_parameters():,}')

In [ ]:
# Cell 4: Configure LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    lora_alpha=128,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({trainable/total*100:.1f}%)')

In [ ]:
# Cell 5: Load and format dataset
from datasets import load_dataset

def format_example(example):
    """Format a single example into the chat template."""
    messages = example['messages']
    # Apply chat template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}

train_dataset = load_dataset('json', data_files=str(TRAIN_PATH), split='train')
val_dataset = load_dataset('json', data_files=str(VAL_PATH), split='train')

train_dataset = train_dataset.map(format_example)
val_dataset = val_dataset.map(format_example)

print(f'Train: {len(train_dataset)} examples')
print(f'Val: {len(val_dataset)} examples')
print(f'\nSample:\n{train_dataset[0]["text"][:500]}...')

In [ ]:
# Cell 6: Train with SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments

OUTPUT_DIR = '/content/roclaw-nav-lora'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # effective batch = 16
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    weight_decay=0.01,
    fp16=True,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=3,
    report_to='none',
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,  # pack multiple short examples into one sequence
)

print('Starting training...')
trainer.train()
print('Training complete!')

In [ ]:
# Cell 7: Evaluate on validation set
import re
import numpy as np
from collections import Counter

FastLanguageModel.for_inference(model)

correct_action = 0
total_eval = 0
speed_errors = []
action_confusion = Counter()

# Evaluate on subset for speed
eval_subset = val_dataset.select(range(min(200, len(val_dataset))))

for example in eval_subset:
    messages = example['messages']
    expected = messages[2]['content']

    # Build input (system + user only)
    input_messages = messages[:2]
    input_text = tokenizer.apply_chat_template(
        input_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(input_text, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.1,
            do_sample=True,
        )

    predicted = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

    # Extract action names
    def get_action(text):
        if text.startswith('TOOLCALL:'):
            try:
                return json.loads(text[9:])
            except:
                return None
        return None

    expected_action = get_action(expected)
    predicted_action = get_action(predicted)

    if expected_action and predicted_action:
        total_eval += 1
        if expected_action['name'] == predicted_action['name']:
            correct_action += 1

            # Calculate speed parameter MAE
            exp_args = expected_action.get('args', {})
            pred_args = predicted_action.get('args', {})
            for key in exp_args:
                if key in pred_args:
                    speed_errors.append(abs(exp_args[key] - pred_args[key]))
        else:
            action_confusion[f'{expected_action["name"]} -> {predicted_action["name"]}'] += 1

accuracy = correct_action / total_eval * 100 if total_eval > 0 else 0
mae = np.mean(speed_errors) if speed_errors else float('nan')

print(f'\n=== Evaluation Results ===')
print(f'Action name accuracy: {correct_action}/{total_eval} ({accuracy:.1f}%)')
print(f'Speed parameter MAE: {mae:.1f}')
print(f'TOOLCALL format compliance: {total_eval}/{len(eval_subset)} ({total_eval/len(eval_subset)*100:.0f}%)')

if action_confusion:
    print(f'\nTop confusions:')
    for pair, count in action_confusion.most_common(5):
        print(f'  {pair}: {count}')

In [ ]:
# Cell 8: Merge LoRA weights
MERGED_DIR = '/content/roclaw-nav-merged'

model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method='merged_16bit',
)
print(f'Merged model saved to {MERGED_DIR}')

In [ ]:
# Cell 9: Export to GGUF
# Q8_0 for quality (inference on decent hardware)
model.save_pretrained_gguf(
    '/content/roclaw-nav-q8_0',
    tokenizer,
    quantization_method='q8_0',
)
print('Q8_0 GGUF exported')

# Q4_K_M for size (edge deployment)
model.save_pretrained_gguf(
    '/content/roclaw-nav-q4km',
    tokenizer,
    quantization_method='q4_k_m',
)
print('Q4_K_M GGUF exported')

# Show file sizes
import os
for d in ['/content/roclaw-nav-q8_0', '/content/roclaw-nav-q4km']:
    for f in os.listdir(d):
        if f.endswith('.gguf'):
            size_mb = os.path.getsize(os.path.join(d, f)) / 1e6
            print(f'{d}/{f}: {size_mb:.0f} MB')

In [ ]:
# Cell 10: Save to Google Drive
import shutil

DRIVE_OUTPUT = Path('/content/drive/MyDrive/roclaw-distill/models')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

# Copy GGUF files to Drive
for src_dir in ['/content/roclaw-nav-q8_0', '/content/roclaw-nav-q4km']:
    for f in os.listdir(src_dir):
        if f.endswith('.gguf'):
            src = os.path.join(src_dir, f)
            dst = DRIVE_OUTPUT / f
            print(f'Copying {src} -> {dst}...')
            shutil.copy2(src, dst)

# Also save training stats
stats = {
    'model': MODEL_NAME,
    'train_examples': train_count,
    'val_examples': val_count,
    'epochs': 3,
    'lora_r': 64,
    'lora_alpha': 128,
    'action_accuracy': accuracy,
    'speed_mae': float(mae) if not np.isnan(mae) else None,
}
with open(DRIVE_OUTPUT / 'training_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print(f'\nAll files saved to {DRIVE_OUTPUT}')
print('Files:', list(DRIVE_OUTPUT.iterdir()))